In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(r"C:\path\to\your\csvs\reconciliation_clean.csv")
df.head()

In [ ]:
df['cost_impact_M'] = df['cost_impact'] / 1e6
df_sorted = df.sort_values('cost_impact_M', ascending=False)
df_sorted[['meg', 'cost_impact_M']]

In [ ]:
colors = ['indianred' if x > 0 else 'seagreen' for x in df_sorted['cost_impact_M']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_sorted['meg'], df_sorted['cost_impact_M'], color=colors)
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Cost Impact ($ Millions)')
ax.set_title('Cost Impact by Medicaid Eligibility Group')
plt.tight_layout()
plt.savefig('cost_impact_by_meg.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['enrollment_variance_pct'] = (df['enrollment_variance'] / df['budget_enrollment']) * 100
df['pmpm_variance_pct'] = (df['pmpm_variance'] / df['budget_pmpm']) * 100
df[['meg', 'enrollment_variance_pct', 'pmpm_variance_pct']]

In [ ]:
offsets = {
    'ABD Adult':               (10, 8),
    'Dual Eligible':           (20, 22),
    'General Adult':           (-10, 10),
    'New Adult Childless':     (-95, 18),
    'New Adult w/Child':       (-95, -20),
    'BD Child':                (65, -8),
    'General Child':           (15, -40),
    'CHIP':                    (10, 8),
    'Dr. D Expansion (IHIP)':  (10, 8),
    'Pharmacy Only':           (10, 8),
    'Vermont Premium Assist.': (-95, -12),
    'Vermont Cost Sharing':    (10, 8),
}

colors2 = ['indianred' if c > 0 else 'seagreen' for c in df['cost_impact_M']]
sizes = [min(max(abs(c) * 10, 50), 500) for c in df['cost_impact_M']]

fig, ax = plt.subplots(figsize=(13, 9))
ax.scatter(df['enrollment_variance_pct'], df['pmpm_variance_pct'], s=sizes, color=colors2, alpha=0.75, edgecolors='black', linewidth=0.5)

for i, row in df.iterrows():
    dx, dy = offsets.get(row['meg'], (10, 8))
    ax.annotate(
        row['meg'],
        (row['enrollment_variance_pct'], row['pmpm_variance_pct']),
        textcoords="offset points", xytext=(dx, dy), fontsize=9,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.6, alpha=0.7) if abs(dx) > 15 or abs(dy) > 15 else None
    )

ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')

# Recalibrated monitor threshold (1 historical stdev): 6.3% enrollment, 8.9% PMPM
ax.axhline(8.9, color='orange', linewidth=0.6, linestyle=':', alpha=0.6)
ax.axhline(-8.9, color='orange', linewidth=0.6, linestyle=':', alpha=0.6)
ax.axvline(6.3, color='orange', linewidth=0.6, linestyle=':', alpha=0.6)
ax.axvline(-6.3, color='orange', linewidth=0.6, linestyle=':', alpha=0.6)

ax.set_xlim(-65, 75)
ax.set_xlabel('Enrollment Variance (%)')
ax.set_ylabel('PMPM Variance (%)')
ax.set_title('Enrollment vs. PMPM Variance by MEG')
plt.tight_layout()
plt.savefig('enroll_var_v_pmpm_var_by_meg.png', dpi=150, bbox_inches='tight')
plt.show()